# Exact inference: variable elimination

_Artificial Intelligence — more_

**Sum out the variables you don't care about, one at a time, multiplying factors as you go.**

To answer a query in a Bayes net exactly, you must sum the joint probability over every variable you did not observe and do not care about.

     Done naively, that sum is astronomically large. Variable elimination makes it tractable by pushing sums inward.

---

This notebook is a practice scaffold for the **Exact inference: variable elimination** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np

# binary factors as 2x2 tables, indexed [A,B] and [B,C].
f1 = np.array([[2.0, 1.0],    # f1(A=0,B=0)=2, f1(0,1)=1
               [1.0, 2.0]])   #              A=1 row
f2 = np.array([[1.0, 2.0],    # f2(B=0,C=0)=1, f2(0,1)=2
               [3.0, 1.0]])   #              B=1 row

# eliminate B: g(A,C) = sum_B f1[A,B] * f2[B,C]  (a matrix product over B)
g = np.einsum("ab,bc->ac", f1, f2)
print("g(A,C) factor after summing out B:")
print(g)
print("g(A=0,C=0) =", g[0, 0], "(expect 2*1 + 1*3 = 5)")

# normalize to a distribution over (A,C) for the query answer
print("normalized:")
print(np.round(g / g.sum(), 3))

## Visualize the data & results

_Sprinkler-Rain-WetGrass network: the grass is wet — what is the posterior probability that it rained?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sprinkler-Rain-WetGrass Bayes net (Russell & Norvig). 0=False, 1=True.
P_C = np.array([0.5, 0.5])
P_S_C = np.array([[0.5, 0.5], [0.9, 0.1]])   # P(Sprinkler | Cloudy)
P_R_C = np.array([[0.8, 0.2], [0.2, 0.8]])   # P(Rain | Cloudy)
P_W = np.zeros((2, 2, 2))                     # P(WetGrass | Sprinkler, Rain)
P_W[0,0] = [1.0, 0.0];  P_W[0,1] = [0.1, 0.9]
P_W[1,0] = [0.1, 0.9];  P_W[1,1] = [0.01, 0.99]

# Eliminate Cloudy and Sprinkler; keep Rain, condition on WetGrass=True.
joint_R = np.zeros(2)
for c in range(2):
    for s in range(2):
        for r in range(2):
            joint_R[r] += P_C[c]*P_S_C[c,s]*P_R_C[c,r]*P_W[s,r,1]
post = joint_R / joint_R.sum()               # [0.292, 0.708]

labels = ["Rain = False", "Rain = True"]
fig, ax = plt.subplots()
bars = ax.bar(labels, post, color=["#ff7b72", "#4ea1ff"])
for b, v in zip(bars, post):
    ax.text(b.get_x()+b.get_width()/2, v, str(round(v,3)), ha="center", va="bottom")
ax.set_title("P(Rain | WetGrass = True) by variable elimination")
plt.show()